# 07 — Spatial animations & envelope analysis (strategic / CEO run)

Loads a finished **strategic (Phase-2 CEO)** run directory, reproduces the spatial
visualisations from notebook 06 via `hotelling.viz.spatial_map`, and adds
strategic-only envelope / collusion / CEO-audit plots from `hotelling.viz.envelopes`.


In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# -- Robustly locate the repository root (dir that contains 'src') --------
def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir():
            return p
    raise RuntimeError(f"Could not find a 'src' directory above {start}")

repo_root = _find_repo_root(Path.cwd())
_src = str(repo_root / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

print(f"repo_root : {repo_root}")

%load_ext autoreload
%autoreload 2
%matplotlib inline

import matplotlib.pyplot as plt
from IPython.display import Image, HTML, display

from hotelling.viz.spatial_map import (
    load_run,
    plot_market_snapshot,
    animate_market,
    interactive_slider,
)
from hotelling.viz import envelopes
from hotelling.simulation.dense_log import DenseLog


In [ ]:
RUNS_ROOT = repo_root / "results" / "strategic_runs" / "runs"
run_dirs = sorted([p for p in RUNS_ROOT.iterdir() if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError(
        f"No strategic run directories under {RUNS_ROOT}.\n"
        "Run  python scripts/run_strategic.py  first."
    )
RUN_DIR = run_dirs[-1]   # most recent; set explicitly to compare specific runs
# RUN_DIR = RUNS_ROOT / "20260615_030900_717fd71a"  # pin a specific run

RUN_DIR = Path(RUN_DIR)
print("Using run:", RUN_DIR)
assert (RUN_DIR / "dense_log_meta.json").exists(), "DenseLog missing — run Step 8 strategic session first."

meta = json.loads((RUN_DIR / "metadata.json").read_text())
print("model:", meta.get("ceo_model"), "| epochs:", meta.get("n_epochs"),
      "| divisions:", meta.get("active_divisions"), "| Δ:", meta.get("deltas_by_chain"))

with (RUN_DIR / "dense_log_meta.json").open() as f:
    _dense_meta = json.load(f)
T_written = int(_dense_meta.get("T_written", _dense_meta.get("T_allocated", 0)))
T_game = int(meta.get("T_game", T_written))
print(f"T_written={T_written}, T_game={T_game}")


## Run metadata

Strategic runs record CEO model, epoch count, group divisions, and Calvano Δ in
`metadata.json`. The DenseLog enables the same spatial pipeline as notebook 06.


In [ ]:
_cfg = yaml.safe_load((RUN_DIR / "config.yaml").read_text()) if (RUN_DIR / "config.yaml").exists() else {}
_agents_cfg = _cfg.get("agents", {})
m_effort = int(_agents_cfg.get("m_effort", 1))

summary = pd.Series({
    "n_firms": meta.get("n_firms"),
    "T_game": meta.get("T_game"),
    "n_epochs": meta.get("n_epochs"),
    "no_ceo": meta.get("no_ceo"),
    "ceo_model": meta.get("ceo_model"),
    "delta_global": meta.get("deltas_by_chain", {}).get("global"),
    "m_effort": m_effort,
    "T_written": T_written,
}, name="value")
display(summary.to_frame())


## Aggregate convergence (Phase 2)


In [ ]:
_agg = pd.read_parquet(RUN_DIR / "aggregate.parquet")
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(_agg["step"], _agg["mean_price"], color="black", lw=2.0, label="Total mean")
_cmap = {"discount": "tab:green", "standard": "tab:blue", "bio": "tab:red"}
for _ct, _c in _cmap.items():
    _col = f"mean_price_{_ct}"
    if _col in _agg.columns:
        ax.plot(_agg["step"], _agg[_col], color=_c, lw=1, label=_ct, alpha=0.8)
ax.set_xlabel("Simulation step"); ax.set_ylabel("Mean price (EUR)")
ax.set_title("Phase-2 price trajectories by chain type")
ax.legend(fontsize=9); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Static spatial snapshots (parity with notebook 06)

Same `plot_market_snapshot` calls — strategic runs write a DenseLog in Step 8.


In [ ]:
plot_market_snapshot(RUN_DIR, t=0, metric="expected_price")


In [ ]:
plot_market_snapshot(RUN_DIR, t=meta.get("T_game", 5000) - 1, metric="expected_price")


In [ ]:
_figures_dir = RUN_DIR / "figures"
_figures_dir.mkdir(parents=True, exist_ok=True)

_t_last = min(T_written - 1, T_game - 1)
_snapshots = [
    (_t_last, "served_demand"),
    (_t_last, "dominant_chain"),
]
for _t, _metric in _snapshots:
    _save = _figures_dir / f"snapshot_t{_t}_{_metric}.png"
    _fig = plot_market_snapshot(
        RUN_DIR, _t, metric=_metric, save_path=_save, point_size_by_demand=True,
    )
    print(f"Saved: {_save}")
    display(_fig)
    plt.close(_fig)


## Spatial animation

Writes `animation_expected_price.gif` into the run directory (gitignored).


In [ ]:
animate_market(RUN_DIR, metric="expected_price", fps=8,
               save_path=RUN_DIR / "animation_expected_price.gif")


In [ ]:
gif_path = RUN_DIR / "animation_expected_price.gif"
if gif_path.exists():
    display(Image(filename=gif_path))
else:
    print("GIF not found — run the animation cell above first.")


## Interactive slider

Requires `%matplotlib widget` or `%matplotlib notebook` for smooth updates.


In [ ]:
try:
    interactive_slider(RUN_DIR, metric="expected_price")
except ImportError as _err:
    print(f"ipywidgets not available ({_err}). Install with: pip install ipywidgets ipympl")


## CEO envelope trajectories

Price bands (p̄ ± Δp), Calvano Δ by chain type, and exploration ε over epochs.


In [ ]:
# CEO price envelopes over epochs: p_bar with shaded [p_bar - delta_p, p_bar + delta_p].
envelopes.plot_envelope_bands(RUN_DIR)
plt.show()


In [ ]:
env = pd.read_parquet(RUN_DIR / "envelopes.parquet")
for grp in sorted(env["group"].unique()):
    envelopes.plot_envelope_bands(RUN_DIR, group=grp)
    plt.show()


In [ ]:
envelopes.plot_delta_by_chain(RUN_DIR)
plt.show()


In [ ]:
envelopes.plot_epsilon_trajectory(RUN_DIR)
plt.show()


## CEO decision audit — what each chain decided and why

Qualitative check: compare each chain's chosen `p_bar` against its marginal cost
and rival prices in the same epoch.


In [ ]:
# Print each CEO's chosen envelope (from envelopes.parquet) alongside its rationale
# (from ceo_decisions.jsonl), per epoch. This is the qualitative check on whether the
# LLM reasoned about the state vs. produced noise.
dec_path = RUN_DIR / "ceo_decisions.jsonl"
decisions = []
if dec_path.exists():
    with dec_path.open() as f:
        decisions = [json.loads(line) for line in f if line.strip()]
dec_df = pd.DataFrame(decisions)
env = pd.read_parquet(RUN_DIR / "envelopes.parquet")

EPOCH = 0  # change to inspect later epochs
print(f"=== CEO decisions @ epoch {EPOCH} ===\n")
if dec_df.empty:
    print("(no CEO decisions — was this a --no-ceo control run?)")
else:
    for _, d in dec_df[dec_df.get("epoch", -1) == EPOCH].iterrows():
        chain = d["chain"]
        rows = env[(env["epoch"] == EPOCH) & (env["chain"] == chain)]
        print(f"--- {chain}  ({d.get('n_groups')} group(s)) ---")
        for _, r in rows.iterrows():
            print(f"   group={r['group']:<16} p_bar={r['p_bar']:.2f}  "
                  f"delta_p={r['delta_p']:.2f}  eps={r['epsilon']:.3f}")
        print(f"   rationale: {str(d.get('rationale',''))[:400]}\n")


The **rationale** text is the CEO's own reasoning (Gemma writes it into the schema's `rationale` field now that thinking is disabled). Compare `p_bar` against the chain's marginal cost and the rivals' prices in the same epoch to judge plausibility.


## Failed LLM calls (silent fallbacks)

Records with `"status": "FAILED"` in `llm_calls.jsonl` indicate Instructor parse failures that fell back to the previous envelope.


In [ ]:
calls_path = RUN_DIR / "llm_calls.jsonl"
failed = []
if calls_path.exists():
    with calls_path.open() as f:
        for line in f:
            try:
                rec = json.loads(line)
            except Exception:
                continue
            if rec.get("status") == "FAILED":
                failed.append(rec)
print(f"{len(failed)} failed CEO call(s).")
for rec in failed[:3]:
    print("  structured error:", rec.get("error_structured"))
    print("  raw (truncated):", str(rec.get("raw_content"))[:200], "\n")
